# Week 3, day 1 (afternoon) — Worksheet 05 SOLUTIONS: reindexing   (L02)

Every cell below was executed in the lab image (pandas 3.0.5) and the quoted
output is what it actually printed — including the error in Q10.

Question 3 is the one to re-read. Reindexing changed the dtype of a column of
whole numbers, and neither the code nor the deck mentions it.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 05 — Reindexing. Run this once.
import pandas as pd

# The lecture's three students.
marks = pd.Series([85, 90, 75], index=["S1", "S2", "S3"], name="Marks")

# The roster the registrar says SHOULD exist -- note S4, who has no marks.
roster = ["S1", "S2", "S3", "S4"]

frame = pd.DataFrame(
    {"Marks": [85, 90, 75], "Grade": ["A", "A", "B"]},
    index=["S1", "S2", "S3"],
)

print(marks)
print()
print("roster:", roster)

PART A — asking for labels you do not have

### Question 1

`S4` -> `NaN`. -> nothing raised and nothing warned.

You asked for a label the data does not contain and got a row for it.
That is not a failure mode, it is the definition of the operation:
reindexing builds a result with *the index you asked for*, filling in
whatever it cannot find.

In [ ]:
print(marks.reindex(roster))
print()
print("nothing raised, nothing warned")

### Question 2

`reindex(["S3", "S1", "S4"])` -> `S3 75.0`, `S1 85.0`, `S4 NaN`. -> index received is exactly the list you passed.

Two operations in one call: the surviving rows came back in *your* order,
not the original order, and the absent label still got a row.

That is what makes reindex the right tool for lining two datasets up
against a shared list of expected keys. You are not asking 'what is in
this data' — you are asserting 'this is the shape the answer must have'.

In [ ]:
out = marks.reindex(["S3", "S1", "S4"])
print(out)
print()
print("index asked for:", ["S3", "S1", "S4"])
print("index received: ", list(out.index))

### Question 3

`int64` -> **`float64`**.

You added no data and removed no data. You named one extra label, and
every mark in the Series changed type.

`NaN` is a floating-point value — there is no integer that means 'missing'
— so the moment one appears, the whole column must widen to hold it. Your
marks are now `85.0`, `90.0`, `75.0`.

This bites when the column is an identifier. Reindex a frame of integer
order IDs against a roster with one gap and every ID becomes a float;
print it and you get `8710.0`. Join on that column and it will not match
the integers on the other side.

In [ ]:
print("before:", marks.dtype)
print("after: ", marks.reindex(roster).dtype)
print()
print(marks.reindex(roster))

### Question 4

`fill_value=0` -> `S4` becomes `0`, and the dtype stays **`int64`**.

No `NaN` was ever created, so nothing had to widen. The dtype problem from
Q3 disappears.

But look at what you asserted. `S4` now has a mark of zero, which reads as
'sat the exam and scored nothing'. The truth is 'did not sit the exam'.
Average these marks and S4 drags the mean down as if it were a real failure.

`fill_value` is a claim about the world, not a formatting option. Use it
when zero is genuinely the right answer — counts of events that did not
happen — and leave `NaN` when the value is unknown, because every
aggregation in Pandas skips `NaN` by default and none of them skip zero.

In [ ]:
filled = marks.reindex(roster, fill_value=0)
print(filled)
print()
print("dtype:", filled.dtype)

# The dtype stayed int64, because no NaN was ever introduced.
# But S4 now has a mark of 0 -- which reads as "sat the exam and scored
# nothing", not "did not sit the exam". NaN was the honest answer.

PART B — reindexing vs selecting

### Question 5

`loc[["S1","S2"]]` and `reindex(["S1","S2"])` -> identical, `.equals()` is `True`.

When every label you name exists, selection and reindexing are the same
operation and there is no reason to prefer one.

Which is exactly why the difference is easy to miss — the two behave
identically on clean data and diverge only on the data that has a problem.

In [ ]:
a = marks.loc[["S1", "S2"]]
b = marks.reindex(["S1", "S2"])
print(a)
print()
print(b)
print()
print("identical:", a.equals(b))

### Question 6

`reindex(["S1", "S2", "S4"])` -> `S1 85.0`, `S2 90.0`, `S4 NaN`.

One label changed from present to absent, and reindex handled it without
comment. Hold this result next to Q10, which is the same list of labels
handed to `.loc`.

In [ ]:
print(marks.reindex(["S1", "S2", "S4"]))
print()
print("reindex invented a row for S4 and did not complain")

### Question 7

`frame.reindex(roster)` -> `S4` gets `NaN` in **both** columns. -> `Marks float64`, `Grade str`.

The whole row was invented, so every column had to supply a value for it.

The two columns paid different prices. `Marks` was `int64` and widened to
`float64`, exactly as in Q3. `Grade` was already a text column, and text
columns can hold a missing marker without changing dtype — so it reads
`str` still. Whether reindexing damages your types depends entirely on
which types you had.

In [ ]:
out = frame.reindex(roster)
print(out)
print()
print(out.dtypes)

# Marks went int64 -> float64 to hold the NaN.
# Grade was already text, so it takes NaN without changing dtype.

### Question 8

`reindex(columns=[...])` -> columns reordered to `['Grade', 'Marks', 'Attendance']`, `Attendance` is all `NaN`. -> `Marks` stayed `85, 90, 75`, still integers.

Same operation, other axis: you named the columns the result must have,
and got a column nobody ever recorded.

Notice `Marks` did **not** become floats this time. No row was added, so no
existing column acquired a `NaN` — only the brand-new `Attendance` column
is missing, and it was never anything else. The dtype promotion in Q3 and
Q7 was caused by holes appearing in *existing* columns, not by reindexing
as such.

In [ ]:
print(frame.reindex(columns=["Grade", "Marks", "Attendance"]))
print()
print("columns:", list(frame.reindex(columns=["Grade", "Marks", "Attendance"]).columns))

### Question 9

`isna()` -> `S4` is the only `True`. -> missing count `1`, missing labels `['S4']`.

This is what reindexing is actually for, and it is worth separating from
the reordering.

Before: `marks` had three rows and looked complete. There was no evidence
in the object that a fourth student was expected. After: the absence is a
row you can count, name, and report.

The pattern generalises. Any time you have a list of things that *should*
be there — a product catalogue, a full calendar of dates, a roster —
reindexing against it converts 'silently absent' into 'present and
missing', which is the only form a gap can be checked in.

In [ ]:
aligned = marks.reindex(roster)
print(aligned.isna())
print()
print("missing count:", aligned.isna().sum())
print("missing labels:", list(aligned.index[aligned.isna()]))

### Question 10

`marks.loc[["S1", "S2", "S4"]]` -> **raises** `KeyError: "['S4'] not in index"`.

The identical list of labels that Q6 accepted. Reindex returned a row of
`NaN`; `.loc` refused outright and named the offending label.

Both behaviours are right for their purpose. `.loc` is a lookup — you are
asserting these rows exist, so a miss is a bug and should stop you.
`reindex` is a reshape — you are describing a target shape, so a miss is
an expected outcome to be filled in.

The practical rule: use `.loc` when a missing label means your assumptions
are wrong, and `reindex` when a missing label is a finding you want to
measure. Choosing on that basis rather than on which one you happen to
remember is most of the skill.

In [ ]:
print(marks.loc[["S1", "S2", "S4"]])